In [2]:
import iris
import os
from pathlib import Path

# Path to the directory containing pp files
data_dir = Path("/home/users/jonathan.lillis/cylc-run/CMEW/CMEW_cmm_test/share/data/cdds/cdds_data/GCModelDev/ESMVal/UKESM1-0-LL_amip-u-dq123_r1i1p1f1/round-1/input/u-dq123/apm/")

# Find all pp files in the directory
pp_files = sorted(data_dir.glob("*.pp"))

if not pp_files:
    print(f"No .pp files found in {data_dir}")
else:
    print(f"Found {len(pp_files)} .pp files\n")

    surface_altitude_coords = {}

    for pp_file in pp_files:
        try:
            print(f"Loading: {pp_file.name}")
            cubes = iris.load(str(pp_file))

            for cube in cubes:
                if cube.name() == "mass_fraction_of_carbon_dioxide_in_air" and cube.coords("surface_altitude"):
                    coord = cube.coord("surface_altitude")
                    surface_altitude_coords[pp_file.name] = {
                        "shape": coord.shape,
                        "dtype": coord.dtype,
                        "units": coord.units,
                        "data_sample": coord.points[:5] if len(coord.points) > 0 else "empty"
                    }
                    print(f"  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape={coord.shape}, dtype={coord.dtype}, units={coord.units}")
                    break
        except Exception as e:
            print(f"  ✗ Error loading {pp_file.name}: {e}")


Found 120 .pp files

Loading: dq123a.pm1983apr.pp


/home/users/jonathan.lillis/.local/lib/python3.12/site-packages/iris/fileformats/rules.py:45: IrisUserWarning: Multiple reference cubes for orography
  warnings.warn(


  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983aug.pp
  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983dec.pp
  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983feb.pp
  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983jan.pp
  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983jul.pp
  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983jun.pp
  ✓ Found surface_altitude for mass_fraction_of_carbon_dioxide_in_air: shape=(144, 192), dtype=float32, units=m
Loading: dq123a.pm1983mar.pp
  ✓ Found sur

In [3]:
import re

# Group surface_altitude signatures by year
year_signatures = {}
missing = []

for pp_file in pp_files:
    m = re.search(r"(19|20)\d{2}", pp_file.name)
    if not m:
        continue
    year = m.group(0)

    cubes = iris.load(str(pp_file))
    target_cube = None
    for cube in cubes:
        if cube.name() == "mass_fraction_of_carbon_dioxide_in_air" and cube.coords("surface_altitude"):
            target_cube = cube
            break

    if target_cube is None:
        missing.append(pp_file.name)
        continue

    coord = target_cube.coord("surface_altitude")
    # Signature for equality checks
    sig = (coord.shape, str(coord.units), coord.points.tobytes())

    year_signatures.setdefault(year, set()).add(sig)

# Check internal consistency per year (all files in same year should match)
internal_consistency = {year: (len(sigs) == 1) for year, sigs in year_signatures.items()}

# Check differences between years (each year should have a different signature)
representative = {year: next(iter(sigs)) for year, sigs in year_signatures.items() if sigs}
unique_across_years = len(set(representative.values())) == len(representative)

print("Internal consistency by year:")
for year in sorted(internal_consistency):
    print(f"  {year}: {'OK' if internal_consistency[year] else 'INCONSISTENT'}")

print(f"\nChanges between years: {'YES' if unique_across_years else 'NO'}")

if missing:
    print("\nFiles missing target cube/surface_altitude:")
    for name in missing:
        print(f"  - {name}")

if all(internal_consistency.values()) and unique_across_years:
    print("\n✓ Confirmed: surface_altitude is internally consistent within each year and changes between years.")
else:
    print("\n✗ Not fully confirmed. See diagnostics above.")

Internal consistency by year:
  1983: OK
  1984: OK
  1985: OK
  1986: OK
  1987: OK
  1988: OK
  1989: OK
  1990: OK
  1991: OK
  1992: OK

Changes between years: YES

✓ Confirmed: surface_altitude is internally consistent within each year and changes between years.


In [22]:
import iris

files = ['/home/users/jonathan.lillis/cylc-run/CMEW/CMEW_cmm_test/share/work/GCModelDev/ESMVal/MOHC/UKESM1-0-LL/amip-u-dr725/r5i1p1f3/AERmon/cdnc/gn/v20260820/cdnc_AERmon_UKESM1-0-LL_amip-u-dr725_r5i1p1f3_gn_318001-318012.nc',
    '/home/users/jonathan.lillis/cylc-run/CMEW/CMEW_cmm_test/share/work/GCModelDev/ESMVal/MOHC/UKESM1-0-LL/amip-u-dr725/r5i1p1f3/AERmon/cdnc/gn/v20260820/cdnc_AERmon_UKESM1-0-LL_amip-u-dr725_r5i1p1f3_gn_318101-318112.nc',
    '/home/users/jonathan.lillis/cylc-run/CMEW/CMEW_cmm_test/share/work/GCModelDev/ESMVal/MOHC/UKESM1-0-LL/amip-u-dr725/r5i1p1f3/AERmon/cdnc/gn/v20260820/cdnc_AERmon_UKESM1-0-LL_amip-u-dr725_r5i1p1f3_gn_318201-318212.nc',
    '/home/users/jonathan.lillis/cylc-run/CMEW/CMEW_cmm_test/share/work/GCModelDev/ESMVal/MOHC/UKESM1-0-LL/amip-u-dr725/r5i1p1f3/AERmon/cdnc/gn/v20260820/cdnc_AERmon_UKESM1-0-LL_amip-u-dr725_r5i1p1f3_gn_318301-318312.nc']
cubes = iris.load(files)
cube

[<iris 'Cube' of number_concentration_of_cloud_liquid_water_particles_in_air / (m-3) (time: 12; atmosphere_hybrid_height_coordinate: 85; latitude: 144; longitude: 192)>]

In [ ]:
import iris

ds902_cube = iris.load_cube('/home/users/jane.mulcahy/public_html/cmm/UKESM13_piC_ch4/nc/awmean/ds902_apy_cdnc_global.nc')
dr725_cube = iris.load_cube('/home/users/jane.mulcahy/public_html/cmm/UKESM13_piC_ch4/nc/awmean/dr725_apy_cdnc_global.nc')

time_coord = ds902_cube.coord("time")
print(f"ds902 {time_coord.units.num2date(time_coord.points[0])}")


ds902 10207440.0
